In [39]:
from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    PromptTemplate,
)
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

In [4]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5, max_tokens=256)

In [5]:
prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content="You are helpful assistant"),
        MessagesPlaceholder("history"),
        ("human", "{question}"),
    ]
)

chat = prompt | llm
history = []


def ask(question):
    reply = chat.invoke({"history": history, "question": question})
    history.append(HumanMessage(question))
    history.append(AIMessage(reply.content))
    return reply.content


print(ask(question="what is the capital of france"))
print(ask(question="what is the population"))


print(history)

The capital of France is Paris.
As of my last update in October 2023, the population of Paris is approximately 2.1 million people within the city limits. However, if you consider the larger metropolitan area, the population is around 12 million. For the most current population figures, please check the latest statistics from reliable sources.
[HumanMessage(content='what is the capital of france', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='what is the population', additional_kwargs={}, response_metadata={}), AIMessage(content='As of my last update in October 2023, the population of Paris is approximately 2.1 million people within the city limits. However, if you consider the larger metropolitan area, the population is around 12 million. For the most current population figures, please check the latest statistics from reliable sour

In [8]:
prompt = PromptTemplate.from_template(
    "Explain this {topic} in an Explain like I am 5 way (ELI5)"
)
parser = StrOutputParser()
chain = prompt | llm | parser
res = chain.invoke({"topic": "volcanoes"})
print(res)

Sure! Imagine the Earth is like a big ball made of different layers, kind of like an onion. Deep inside the Earth, it's really hot, and there's melted rock called magma. Sometimes, this magma wants to come out because it gets too crowded down there.

When the magma pushes its way up through cracks in the Earth’s surface, it can create a volcano. Think of a volcano like a big mountain that can spit out lava, ash, and gases when it erupts. 

When the volcano erupts, it’s like a big, exciting explosion! The lava flows down the sides and can make new land. After a while, the volcano might stop erupting and can become quiet, like a sleeping giant. 

So, volcanoes are nature’s way of letting out some of that hot, melted rock from deep inside the Earth!


In [11]:
# Take a input and run these in parallel

# Run Summary
summary_prompt = PromptTemplate.from_template("Summarize in one sentence: {text}")
# Run Translation
translation_prompt = PromptTemplate.from_template("Translate to French: {text}")
# Run Sentiment
sentiment_prompt = PromptTemplate.from_template(
    "Is the sentiment of this text positive, negative or neutral? One word only: {text}"
)

chain = RunnableParallel(
    summary=summary_prompt | llm | StrOutputParser(),
    sentiment=sentiment_prompt | llm | StrOutputParser(),
    translation=translation_prompt | llm | StrOutputParser(),
)

chain.invoke({"text": "I love this product"})

{'summary': 'I am very fond of this product.',
 'sentiment': 'Positive',
 'translation': "J'adore ce produit."}

In [24]:
template = "Tell me a {adjective} joke about {topic}"


def format_prompt(inputs):
    return template.format(adjective=inputs["adjective"], topic=inputs["topic"])


joke_chain = RunnableLambda(format_prompt) | llm | StrOutputParser()

res = joke_chain.invoke({"adjective": "silly", "topic": "robots"})
print(res)

Why did the robot go on a diet?

Because it had too many bytes!


In [ ]:
@tool
def get_population(country: str) -> str:
    """
    Return the population of a country
    """
    data = {"italy": "about 59 million", "france": "about 68 million"}
    return data.get(country.lower(), "unknown")


tools = {"get_population": get_population}
llm_with_tools = llm.bind_tools(list(tools.values()))

messages = [HumanMessage("What is the population of italy?")]

while True:
    ai = llm_with_tools.invoke(messages)

    print("AI: ", ai)
    print("TOOL CALLS: ", ai.tool_calls)

    messages.append(ai)

    if not ai.tool_calls:
        break
    for call in ai.tool_calls:
        print(f"  tool: {call['name']}  args: {call['args']}")
        result = tools[call["name"]].invoke(call["args"])
        messages.append(ToolMessage(content=result, tool_call_id=call["id"]))


AI:  content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 51, 'total_tokens': 65, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9799b6afbf', 'id': 'chatcmpl-EZn6agd2dp1Zc5w7IhgytpPGRQvqB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a14761-b24a-7b20-96c5-a73ab06a3cae-0' tool_calls=[{'name': 'get_population', 'args': {'country': 'Italy'}, 'id': 'call_ookgpUJArDZyxsnUQIXcToUd', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 51, 'output_tokens': 14, 'total_tokens': 65, 'input_token_details': {'audio': 0, '

In [51]:
print(messages[-1])


content='The population of Italy is approximately 59 million.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 77, 'total_tokens': 88, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9799b6afbf', 'id': 'chatcmpl-EZn6at8GXowoAjkXTKfZxdNdUlNKe', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a14761-b5cd-75c0-a5b0-76d8b3a0530e-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 77, 'output_tokens': 11, 'total_tokens': 88, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [44]:
tools

{'get_population': StructuredTool(name='get_population', description='Return the population of a country', args_schema=<class 'langchain_core.utils.pydantic.get_population'>, func=<function get_population at 0x117749b10>)}